In [0]:
# -- GeoBrix: lightweight tier (option-1, default). Installed here so nothing is
#    assumed pre-staged. For the heavyweight tier (option-2 below) attach the
#    GeoBrix JAR + GDAL init script to a classic x86 cluster.
#
# Resilient two-step install (for WHL refreshes in the SAME session) — keep BOTH
# lines; do not collapse to one. Step 1 force-reinstalls WITHOUT deps so a freshly
# rebuilt wheel's bytes always replace the cached install even when the version
# string is unchanged (pip would otherwise treat same-version as "already satisfied"
# and skip it). Step 2 installs the EXTRAS without --no-deps so light/stac/vizx/
# overture dependencies resolve — a bare --no-deps install drops the extras and
# surfaces as ModuleNotFound at import. A `%restart_python` (next cell) then loads
# the fresh bytes. Same pattern as the eo-series / h3-rasterize examples.
%pip install --quiet --disable-pip-version-check --force-reinstall --no-deps "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.4.0-py3-none-any.whl"
%pip install --quiet "geobrix[light,stac,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.4.0-py3-none-any.whl"
%pip install --quiet rich

In [0]:
%restart_python

In [0]:
# ============================================================
# USER SETTINGS  —  edit these (everything below is wired off them)
# ============================================================

# Unity Catalog: a Volume named 'data' must already exist under catalog/schema.
catalog_name = "geospatial_docs"
schema_name = "geobrix"

# Toggles (also overridable per-notebook in a cell right after %run ./config_nb):
FORCE_REBUILD = False      # True -> re-create tables / re-download / re-tile (skip-guards off)
INTERACTIVE_PLOTS = False  # False -> fast static maps that render on GitHub; True -> interactive maps

# Area of interest (San Francisco). Change to retarget the series.
SF_AOI_BBOX = (-123.0, 37.0, -122.0, 38.0)        # full 1x1-deg quad (minx, miny, maxx, maxy; EPSG:4326)
SF_CITY_BBOX = (-122.52, 37.70, -122.35, 37.83)   # tighter city sub-bbox (demo-friendly volumes)

In [0]:
# -- databricks + delta + spark functions
from delta.tables import *
from pyspark.databricks.sql import functions as DBF
from pyspark.sql import functions as F
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# -- other imports
from datetime import datetime
from databricks.labs.gbx.stac import StacClient
from databricks.labs.gbx.sample.overture import OvertureClient

import os
import pandas as pd
import pathlib
import warnings

warnings.simplefilter("ignore")

In [ ]:
# Overture (GeoParquet -> Volume; NB01) + Planetary-Computer STAC clients (search/download
# + COG catalog; NB02-03). STAC API docs: databrickslabs.github.io/geobrix/docs/api/stac
overture = OvertureClient()      # Overture Maps static STAC catalog; latest release resolved by default
stac_client = StacClient()       # used in NB03 to STAC-catalog the COGs we generate

In [ ]:
# -- Spark conf tuning, guarded for Serverless --
# Serverless forbids runtime spark.conf.set; set_conf_safe() no-ops there (AQE
# handles partitioning). On classic clusters it applies high-parallelism tuning.
def set_conf_safe(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception as e:
        print(f"... skipping spark.conf.set({key}) [Serverless?]: {type(e).__name__}")
        return False

set_conf_safe("spark.sql.adaptive.coalescePartitions.enabled", "false")
set_conf_safe("spark.sql.shuffle.partitions", 512)

In [0]:
# -- GeoBrix tier selection (default: lightweight) --
# option-1: lightweight tier (pure Python / PySpark, runs on Serverless) -- DEFAULT
from databricks.labs.gbx.pyrx import functions as rx     # rx.rst_*  (raster)
from databricks.labs.gbx.pyvx import functions as vx     # vx.st_*   (vector)

# option-2: heavyweight tier (Scala JAR + GDAL init script on a classic x86 cluster)
# from databricks.labs.gbx.rasterx import functions as rx
# from databricks.labs.gbx.vectorx import functions as vx

rx.register(spark)    # registers gbx_rst_*, gbx_rst_xyzpyramid, gbx_rst_cog_convert, gbx_pmtiles_agg, ...
vx.register(spark)    # registers gbx_st_asmvt, gbx_st_asmvt_pyramid, ...

# -- light readers/writers (gtiff_gbx, binaryFile patterns, pmtiles writer) --
from databricks.labs.gbx.ds.register import register
register(spark)

In [0]:
# -- visualization: PMTiles + COG viewers (net-new), plus the raster + vector helpers.
#    plot_static / plot_interactive back the INTERACTIVE_PLOTS toggle (static images for
#    GitHub by default; interactive MapLibre pan/zoom maps when True). See the helper cells below.
from databricks.labs.gbx.vizx import (
    plot_pmtiles, plot_cog, plot_raster, plot_file, plot_static, plot_interactive,
    cells_as_gdf, simplify_tiles_from_source,
)
from databricks.labs.gbx.pmtiles import pmtiles_info       # driver-side PMTiles header inspector

In [ ]:
# Apply the catalog/schema chosen in USER SETTINGS above.
sql(f"USE CATALOG {catalog_name}")
sql(f"CREATE DATABASE IF NOT EXISTS {schema_name}")
sql(f"USE DATABASE {schema_name}")
print(f"... catalog: '{catalog_name}' (USE)")
print(f"... schema: '{schema_name}' (CREATE / USE)")

In [ ]:
ETL_DIR = f"/Volumes/{catalog_name}/{schema_name}/data"  # <- Volume ('data') must exist
HELIOS_DIR = f"{ETL_DIR}/helios"
dbutils.fs.mkdirs(HELIOS_DIR)
os.environ["ETL_DIR"] = ETL_DIR
os.environ["HELIOS_DIR"] = HELIOS_DIR

print(f"... ETL_DIR: '{ETL_DIR}'")
print(f"... HELIOS_DIR: '{HELIOS_DIR}' (MKDIRS)")
print(f"... SF_CITY_BBOX: {SF_CITY_BBOX}")

In [ ]:
# Solar-walkthrough helper functions (series-only — NOT the GeoBrix API):
#   solar_score                     — south-facing-slope suitability score in [0,1]
#   finalize_delta                  — idempotent managed-Delta materializer (the
#                                     Serverless-safe stand-in for .cache(); self-heals
#                                     on schema drift)
#   show_pmtiles / show_cog / show_raster — demo viewers gated by INTERACTIVE_PLOTS
#                                     (static image by default; interactive MapLibre when True)
def solar_score(slope_col="slope_deg", aspect_col="aspect_deg"):
    """Column expression: solar-suitability score in [0, 1] for a roof/terrain facet.

    Favors gentle slopes (best near a target tilt for SF latitude ~30 deg) and
    south-facing aspect (180 deg). Aspect is GDAL convention (0=N, 90=E, 180=S,
    270=W). This is a didactic score, not a production PV yield model.
    """
    target_tilt = 30.0
    slope_term = 1.0 - (F.abs(F.col(slope_col) - target_tilt) / 90.0)
    # cos of (aspect - south) folds 0..360 into a south-preference in [0, 1]
    aspect_term = (F.cos(F.radians(F.col(aspect_col) - 180.0)) + 1.0) / 2.0
    return F.greatest(F.lit(0.0), slope_term) * aspect_term

In [ ]:
def finalize_delta(df, tbl_name, do_display=True):
    """Idempotent managed-Delta materializer (Serverless-safe stand-in for cache()).

    Writes `df` to managed table `tbl_name` once; re-reads on subsequent runs unless
    FORCE_REBUILD is True. If the existing table's columns no longer match `df` — a
    schema change while iterating on a notebook — it is transparently rewritten so a
    stale schema never lingers (skipping would return the old columns and break
    downstream selects). Returns the table DataFrame.
    """
    if FORCE_REBUILD:
        sql(f"DROP TABLE IF EXISTS {tbl_name}")
    elif spark.catalog.tableExists(tbl_name):
        # Self-heal on schema drift: persisted columns differ from df's -> stale table.
        if [f.name for f in spark.table(tbl_name).schema] != [f.name for f in df.schema]:
            print(f"... schema changed for {tbl_name} -> rewriting (was stale)")
            sql(f"DROP TABLE IF EXISTS {tbl_name}")
    if not spark.catalog.tableExists(tbl_name):
        df.write.mode("overwrite").saveAsTable(tbl_name)
        print(f"... wrote table {tbl_name} ({spark.table(tbl_name).count():,} rows)")
    else:
        print(f"... table {tbl_name} exists (skip; FORCE_REBUILD=False)")
    out = spark.table(tbl_name)
    if do_display:
        out.printSchema()
    return out

In [0]:
def show_pmtiles(path, **kw):
    """Print the PMTiles header, then render through the INTERACTIVE_PLOTS toggle.

    INTERACTIVE_PLOTS=False (default): plot_pmtiles(path, max_embed_mb=0, ...) forces
    the GitHub-renderable static image. True: the interactive MapLibre map (default
    plot_pmtiles path, base64-embedded in-browser FileSource).
    """
    info = pmtiles_info(path)
    print(f"... pmtiles: type={info.get('tile_type')} "
          f"zoom={info.get('min_zoom')}-{info.get('max_zoom')} "
          f"bounds={info.get('bounds')}")
    if INTERACTIVE_PLOTS:
        return plot_pmtiles(path, **kw)               # interactive MapLibre (default)
    return plot_pmtiles(path, max_embed_mb=0, **kw)   # static render (max_embed_mb=0)

def show_cog(path, **kw):
    """Render a COG. plot_cog is static (rasterio overview read over a contextily
    basemap) in both modes; the toggle is honored for API symmetry with show_pmtiles."""
    return plot_cog(path, **kw)

def show_raster(df_or_path, **kw):
    """Render a raster/vector overlay through the toggle: plot_static (False, default)
    or plot_interactive (True, MapLibre). Accepts a Spark DataFrame of cells/geoms or a
    file path, matching the underlying vizx helpers."""
    if INTERACTIVE_PLOTS:
        return plot_interactive(df_or_path, **kw)
    return plot_static(df_or_path, **kw)